# 🔎 ICU Patient Similarity Search — MIMIC-IV (Kaggle)

Turns the **first 24 h of every ICU stay** into an embedding and indexes all past patients with **FAISS**. For a new
patient it finds the most similar past patients and summarises **what happened to them**: death, long ICU stay,
ventilation, vasopressors, AKI.

**Before Run All**
1. Accelerator **GPU T4** · Internet **On** (installs `faiss-cpu`).
2. Add-ons → **Google Cloud Services** → attach the Google account with PhysioNet BigQuery access → tick
   **BigQuery** and **Cloud Storage**.
3. Set `project_id` in the Config cell. The `mimiciv_derived` (mimic-code) dataset must be visible in BigQuery.

**Resume:** query results (Parquet) and encoder checkpoints (model + optimizer + position inside the epoch) are kept in
a private bucket (`gs://<project_id>-mimic-ckpt/icu-patient-similarity-search/`). If the session stops, **Run All**
again.

Pipeline (~15–30 min): BigQuery → features → encoder → FAISS index → retrieval evaluation vs raw-feature / PCA search →
exact vs IVF-PQ speed → demo query with a hypothetical patient → report.
To try your own query, edit `DEMO_PATIENT` in the demo cell (values you leave out are set to typical values).

## 1 · Setup
BigQuery/GCS clients (already on Kaggle; this only fills gaps) and GPU check.

In [ ]:
!pip install -q google-cloud-bigquery google-cloud-bigquery-storage google-cloud-storage db-dtypes pyarrow faiss-cpu 2>&1 | tail -1
import torch, platform
print("python", platform.python_version(), "| torch", torch.__version__, "| GPU:",
   torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")

## 2 · Project code
These cells write the package to `/kaggle/working/repo`. No need to edit them; the steps further down call this code.

In [ ]:
!mkdir -p /kaggle/working/repo/src

In [ ]:
%%writefile /kaggle/working/repo/src/__init__.py


In [ ]:
%%writefile /kaggle/working/repo/src/bq.py
"""SQL -> Parquet, cached in the bucket.

* BigQuery: `EXPORT DATA` writes the query result straight into your GCS bucket as Parquet. That is much faster
  than pulling millions of rows through the API, and the result stays cached there: a re-run (or a new Kaggle
  session) skips BigQuery completely and only downloads the files.
* Tests: the same BigQuery SQL is transpiled to DuckDB (sqlglot) and run on small synthetic tables.

SQL uses placeholders such as `{hosp}` / `{icu}` / `{derived}` / `{eicu}` for the PhysioNet datasets.
Only numeric / text columns are exported (times are turned into hours inside SQL), so Parquet types are simple.
"""
from __future__ import annotations

import hashlib
import json
import logging
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd

log = logging.getLogger(__name__)


class BigQuerySource:
    kind = "bigquery"

    def __init__(self, cfg, store, datasets: dict):
        """datasets: {placeholder: (override, [candidate dataset names], probe_table)}"""
        from .gcp import bq_client, resolve_dataset
        self.client = bq_client(cfg.project_id)
        self.store = store
        self.ds = {k: resolve_dataset(self.client, o, c, p) for k, (o, c, p) in datasets.items()}
        for k, v in self.ds.items():
            log.info("{%s} -> %s", k, v)

    def fmt(self, sql: str) -> str:
        return sql.format(**{k: f"`{v}`" for k, v in self.ds.items()})

    def export(self, sql: str, key: str):
        from .store import GCSStore
        sql = self.fmt(sql)
        if isinstance(self.store, GCSStore):
            uri = self.store.uri(f"{key}/part-*.parquet")
            q = f"EXPORT DATA OPTIONS(uri='{uri}', format='PARQUET', compression='SNAPPY', overwrite=true) AS\n{sql}"
            try:
                job = self.client.query(q)
                job.result()
                log.info("BigQuery processed %.2f GB -> %s", (job.total_bytes_processed or 0) / 1e9, uri)
                return
            except Exception as e:                     # e.g. the token may not write to GCS -> stream instead
                log.warning("EXPORT DATA failed (%s). Streaming the result through the Storage API instead.",
                            str(e)[:200])
        self._stream(sql, key)

    def _stream(self, sql: str, key: str, rows_per_file: int = 2_000_000):
        """Run the query and write the result in Parquet parts (Storage Read API if allowed, else REST pages)."""
        import pyarrow as pa
        import pyarrow.parquet as pq
        job = self.client.query(sql)
        it = job.result(page_size=100_000)
        log.info("BigQuery processed %.2f GB, %s rows", (job.total_bytes_processed or 0) / 1e9, it.total_rows)
        bqs = None
        try:
            from google.cloud import bigquery_storage
            from .gcp import credentials
            bqs = bigquery_storage.BigQueryReadClient(credentials=credentials("bigquery"))
        except Exception:
            pass
        tmp = Path("/tmp") / key.replace("/", "_")
        tmp.mkdir(parents=True, exist_ok=True)
        buf, n, part = [], 0, 0

        def flush():
            nonlocal buf, n, part
            if buf:
                f = tmp / f"part-{part:05d}.parquet"
                pq.write_table(pa.concat_tables(buf, promote_options="default"), f, compression="snappy")
                self.store.put(f, f"{key}/{f.name}")
                f.unlink()
                buf, n, part = [], 0, part + 1
        try:
            batches = it.to_arrow_iterable(bqstorage_client=bqs)
        except TypeError:
            batches = it.to_arrow_iterable()
        for b in batches:
            t = b if isinstance(b, pa.Table) else pa.Table.from_batches([b])
            buf.append(t)
            n += t.num_rows
            if n >= rows_per_file:
                flush()
        flush()
        if part == 0:                                   # empty result: still write one (empty) file
            f = tmp / "part-00000.parquet"
            it.to_arrow().to_pandas().to_parquet(f, index=False)
            self.store.put(f, f"{key}/{f.name}")

    def df(self, sql: str) -> pd.DataFrame:
        from .gcp import query_df
        return query_df(self.client, self.fmt(sql))


class DuckDBSource:
    """Runs the BigQuery SQL on DuckDB (unit tests / CI with synthetic tables)."""
    kind = "duckdb"

    def __init__(self, con, schemas: dict, store):
        self.con, self.ds, self.store = con, schemas, store

    def fmt(self, sql: str) -> str:
        import sqlglot
        return sqlglot.transpile(sql.format(**self.ds), read="bigquery", write="duckdb")[0]

    def export(self, sql: str, key: str):
        tmp = Path(self.store.uri(f"{key}/part-000.parquet"))
        tmp.parent.mkdir(parents=True, exist_ok=True)
        self.con.execute(f"COPY ({self.fmt(sql)}) TO '{tmp.as_posix()}' (FORMAT PARQUET)")

    def df(self, sql: str) -> pd.DataFrame:
        return self.con.execute(self.fmt(sql)).df()


def materialize(src, store, name: str, sql: str, cache_dir) -> list[Path]:
    """Run `sql` once (result cached in the bucket under data/<name>-<hash>/) and return local Parquet paths."""
    key = f"data/{name}-{hashlib.sha1(src.fmt(sql).encode()).hexdigest()[:10]}"
    if store.exists(f"{key}/_SUCCESS"):
        log.info("[%s] cached in bucket (%s) -> BigQuery skipped", name, store.uri(key))
    else:
        t = time.time()
        src.export(sql, key)
        store.put_text(json.dumps({"seconds": round(time.time() - t, 1)}), f"{key}/_SUCCESS")
        log.info("[%s] query done in %.0f s", name, time.time() - t)
    parts = sorted(k for k in store.list(key) if k.endswith(".parquet"))
    local = Path(cache_dir) / key
    local.mkdir(parents=True, exist_ok=True)

    def fetch(k):
        dst = local / Path(k).name
        if not dst.exists():
            store.get(k, dst)
        return dst
    with ThreadPoolExecutor(8) as ex:
        paths = list(ex.map(fetch, parts))
    log.info("[%s] %d parquet file(s) ready in %s", name, len(paths), local)
    return paths


def read(paths, columns=None) -> pd.DataFrame:
    import pyarrow.parquet as pq
    import pyarrow as pa
    tabs = [pq.read_table(p, columns=columns) for p in paths]
    tabs = [t for t in tabs if t.num_rows] or tabs[:1]
    return pa.concat_tables(tabs, promote_options="default").to_pandas() if tabs else pd.DataFrame()


def n_rows(paths) -> int:
    import pyarrow.parquet as pq
    return sum(pq.ParquetFile(p).metadata.num_rows for p in paths)


In [ ]:
%%writefile /kaggle/working/repo/src/cohort.py
"""MIMIC-IV ICU stays (adult, >= 24 h): first-24-h data as input, outcomes AFTER the first 24 h as targets.

Outcomes: in-hospital death, ICU stay > 7 days, invasive ventilation after 24 h, vasopressor after 24 h,
AKI stage >= 2 between 24 h and 7 days (KDIGO, mimic-code). All from mimic-code `derived` concepts.
"""
from __future__ import annotations

OUTCOMES = ["death", "long_icu_stay", "vent_after_24h", "vaso_after_24h", "aki2_day2_7"]
LABS = ["hemoglobin_min", "platelets_min", "wbc_max", "bicarbonate_min", "bun_max", "creatinine_max", "sodium_min",
        "potassium_max", "aniongap_max", "glucose_max", "inr_max", "bilirubin_total_max", "albumin_min"]
SOFA = ["sofa", "respiration", "coagulation", "liver", "cardiovascular", "cns", "renal"]
VITALS = {"hr": "heart_rate", "sbp": "sbp", "mbp": "mbp", "rr": "resp_rate", "temp": "temperature", "spo2": "spo2",
          "glucose": "glucose"}


def _stays(cfg) -> str:
    return f"""
WITH s AS (
  SELECT d.subject_id, d.hadm_id, d.stay_id, d.icu_intime AS intime, d.admission_age, d.gender, i.first_careunit,
    a.admission_type, d.los_icu, d.hospital_expire_flag
  FROM {{derived}}.icustay_detail d
  JOIN {{icu}}.icustays i ON i.stay_id = d.stay_id
  JOIN {{hosp}}.admissions a ON a.hadm_id = d.hadm_id
  WHERE d.admission_age >= {cfg.min_age} AND d.los_icu >= {cfg.seq_hours / 24}
)"""


def cohort_sql(cfg) -> str:
    h = cfg.seq_hours
    return _stays(cfg) + f"""
, vent AS (
  SELECT s.stay_id, 1 AS x FROM s JOIN {{derived}}.ventilation v ON v.stay_id = s.stay_id
   AND v.ventilation_status = 'InvasiveVent' AND v.endtime > DATETIME_ADD(s.intime, INTERVAL {h} HOUR)
  GROUP BY 1
), vaso AS (
  SELECT s.stay_id, 1 AS x FROM s JOIN {{derived}}.vasoactive_agent v ON v.stay_id = s.stay_id
   AND v.endtime > DATETIME_ADD(s.intime, INTERVAL {h} HOUR)
  GROUP BY 1
), aki AS (
  SELECT s.stay_id, 1 AS x FROM s JOIN {{derived}}.kdigo_stages k ON k.stay_id = s.stay_id AND k.aki_stage >= 2
   AND k.charttime > DATETIME_ADD(s.intime, INTERVAL {h} HOUR) AND k.charttime <= DATETIME_ADD(s.intime, INTERVAL 7 DAY)
  GROUP BY 1
)
SELECT s.subject_id, s.stay_id, s.admission_age AS age, IF(s.gender = 'F', 1, 0) AS female, s.first_careunit,
  s.admission_type,
  {", ".join(f"f.{c}" for c in SOFA)},
  {", ".join(f"l.{c}" for c in LABS)},
  g.gcs_min, u.urineoutput, w.weight, ch.charlson_comorbidity_index AS charlson,
  s.hospital_expire_flag AS death, IF(s.los_icu > 7, 1, 0) AS long_icu_stay,
  IFNULL(vent.x, 0) AS vent_after_24h, IFNULL(vaso.x, 0) AS vaso_after_24h, IFNULL(aki.x, 0) AS aki2_day2_7
FROM s
LEFT JOIN {{derived}}.first_day_sofa f ON f.stay_id = s.stay_id
LEFT JOIN {{derived}}.first_day_lab l ON l.stay_id = s.stay_id
LEFT JOIN {{derived}}.first_day_gcs g ON g.stay_id = s.stay_id
LEFT JOIN {{derived}}.first_day_urine_output u ON u.stay_id = s.stay_id
LEFT JOIN {{derived}}.first_day_weight w ON w.stay_id = s.stay_id
LEFT JOIN {{derived}}.charlson ch ON ch.hadm_id = s.hadm_id
LEFT JOIN vent ON vent.stay_id = s.stay_id
LEFT JOIN vaso ON vaso.stay_id = s.stay_id
LEFT JOIN aki ON aki.stay_id = s.stay_id
"""


def hourly_sql(cfg) -> str:
    aggs = ",\n  ".join(f"AVG(v.{c}) AS {k}" for k, c in VITALS.items())
    return _stays(cfg) + f"""
SELECT s.stay_id, CAST(FLOOR(DATETIME_DIFF(v.charttime, s.intime, MINUTE) / 60) AS INT64) AS h,
  {aggs}
FROM s JOIN {{derived}}.vitalsign v ON v.stay_id = s.stay_id
 AND v.charttime >= s.intime AND v.charttime < DATETIME_ADD(s.intime, INTERVAL {cfg.seq_hours} HOUR)
GROUP BY 1, 2
"""


In [ ]:
%%writefile /kaggle/working/repo/src/config.py
from __future__ import annotations

from dataclasses import asdict, dataclass


@dataclass
class Config:
    # --- data: MIMIC-IV ICU stays (>= 24 h), first 24 h only, full cohort ---
    hosp_dataset: str = ""
    icu_dataset: str = ""
    derived_dataset: str = ""
    min_age: int = 18
    seq_hours: int = 24
    split: tuple = (0.7, 0.15, 0.15)       # by patient: train = the searchable "past patients" database
    seed: int = 42

    # --- encoder (GRU + static -> embedding, multi-task heads), resumable ---
    emb_dim: int = 64
    gru_hidden: int = 64
    static_hidden: int = 128
    dropout: float = 0.2
    batch_size: int = 512
    lr: float = 1e-3
    epochs: int = 40
    patience: int = 6

    # --- retrieval ---
    k: int = 50                            # neighbours used to estimate a query's outcome risk
    k_lift: int = 10
    ivf_nlist: int = 256                   # approximate FAISS index (speed comparison)
    ivf_pq_m: int = 16
    n_boot: int = 200

    # --- outputs + checkpoints (private GCS bucket, auto-created) ---
    work_dir: str = "/kaggle/working/similarity"
    cache_dir: str = "/tmp/similarity_cache"
    project_id: str = ""
    bucket: str = ""
    bucket_location: str = "US"
    gcs_prefix: str = "icu-patient-similarity-search"
    run_name: str = "main"
    local_store: str = ""
    ckpt_every_min: float = 5.0
    device: str = ""

    def datasets(self) -> dict:
        return {"hosp": (self.hosp_dataset, ["mimiciv_3_1_hosp", "mimiciv_hosp", "mimiciv_2_2_hosp"], "admissions"),
                "icu": (self.icu_dataset, ["mimiciv_3_1_icu", "mimiciv_icu", "mimiciv_2_2_icu"], "icustays"),
                "derived": (self.derived_dataset, ["mimiciv_3_1_derived", "mimiciv_derived", "mimiciv_2_2_derived"],
                            "icustay_detail")}

    def to_dict(self):
        return asdict(self)


In [ ]:
%%writefile /kaggle/working/repo/src/features.py
"""Static first-day table + hourly vital-sign sequence -> model inputs. `Prep` keeps the training statistics so that a
new (or hypothetical) patient can be encoded the same way (used by the demo query)."""
from __future__ import annotations

import logging
from dataclasses import dataclass

import numpy as np
import pandas as pd

from .cohort import LABS, OUTCOMES, SOFA, VITALS

log = logging.getLogger(__name__)
RANGES = {"hr": (20, 250), "sbp": (40, 260), "mbp": (20, 200), "rr": (2, 70), "temp": (30, 43), "spo2": (50, 100),
          "glucose": (20, 1000)}
NUM = ["age", "female", "gcs_min", "urineoutput", "weight", "charlson"] + SOFA + LABS
SKEW = ["urineoutput", "bun_max", "creatinine_max", "wbc_max", "inr_max", "bilirubin_total_max", "glucose_max"]


class Prep:
    def fit(self, coh: pd.DataFrame, cube: dict):
        n = self._num(coh)
        self.med = n.median()
        self.miss_cols = list(n.columns[n.isna().mean() > 0.05])
        z = n.fillna(self.med)
        self.mu, self.sd = z.mean(), z.std().replace(0, 1).fillna(1)
        self.cats = {c: coh[c].fillna("unknown").value_counts().index[:10].tolist()
                     for c in ("first_careunit", "admission_type")}
        self.seq = {}
        for k in VITALS:
            v = cube[k][~np.isnan(cube[k])]
            self.seq[k] = (float(np.median(v)) if len(v) else 0.0, float(v.mean()) if len(v) else 0.0,
                           float(v.std()) if len(v) > 1 and v.std() > 0 else 1.0)
        return self

    @staticmethod
    def _num(coh):
        n = coh.reindex(columns=NUM).apply(pd.to_numeric, errors="coerce").astype(float)
        for c in SKEW:
            n[c] = np.log1p(n[c].clip(lower=0))
        return n

    def static(self, coh: pd.DataFrame) -> np.ndarray:
        n = self._num(coh)
        flags = n[self.miss_cols].isna().astype(float).add_suffix("_missing")
        z = ((n.fillna(self.med) - self.mu) / self.sd).clip(-6, 6)
        oh = [pd.get_dummies(pd.Categorical(coh[c].fillna("unknown").where(coh[c].isin(v), "other"),
                                            categories=v + ["other"]), prefix=c, dtype=float)
              for c, v in self.cats.items()]
        return pd.concat([z.reset_index(drop=True), flags.reset_index(drop=True)] +
                         [o.reset_index(drop=True) for o in oh], axis=1).to_numpy(np.float32)

    def sequence(self, cube: dict) -> np.ndarray:
        vals, mask = [], []
        for k in VITALS:
            a = cube[k]
            m = ~np.isnan(a)
            f = pd.DataFrame(a).ffill(axis=1).bfill(axis=1).to_numpy()
            med, mu, sd = self.seq[k]
            f = np.where(np.isnan(f), med, f)
            vals.append(np.clip((f - mu) / sd, -6, 6))
            mask.append(m.astype(float))
        return np.stack(vals + mask, -1).astype(np.float32)


def cube_from_long(long: pd.DataFrame, stays: np.ndarray, hours: int) -> dict:
    pos = pd.Series(np.arange(len(stays)), index=stays)
    long = long[long.stay_id.isin(pos.index) & long.h.between(0, hours - 1)]
    i, t = pos[long.stay_id].to_numpy(), long.h.to_numpy().astype(int)
    out = {}
    for k in VITALS:
        a = np.full((len(stays), hours), np.nan)
        v = pd.to_numeric(long[k], errors="coerce").to_numpy(float)
        lo, hi = RANGES[k]
        a[i, t] = np.where((v >= lo) & (v <= hi), v, np.nan)
        out[k] = a
    return out


@dataclass
class Data:
    static: np.ndarray
    seq: np.ndarray
    Y: np.ndarray                   # [n, len(OUTCOMES)]
    part: np.ndarray
    raw: np.ndarray                 # baseline feature space: static + per-vital 24-h mean/min/max (standardised)
    careunit: np.ndarray
    prep: Prep

    def idx(self, p):
        return np.flatnonzero(self.part == p)


def build(coh: pd.DataFrame, hourly: pd.DataFrame, cfg) -> Data:
    coh = coh.drop_duplicates("stay_id").reset_index(drop=True)
    stays = coh.stay_id.to_numpy()
    cube = cube_from_long(hourly, stays, cfg.seq_hours)
    u = (pd.util.hash_pandas_object(coh.subject_id.astype(str) + f"|{cfg.seed}", index=False).to_numpy() % 10_000) / 10_000
    a, b = cfg.split[0], cfg.split[0] + cfg.split[1]
    part = np.where(u < a, "train", np.where(u < b, "val", "test"))
    tr = part == "train"
    prep = Prep().fit(coh[tr], {k: v[tr] for k, v in cube.items()})
    S, Q = prep.static(coh), prep.sequence(cube)
    nv = len(VITALS)
    agg = np.concatenate([Q[:, :, :nv].mean(1), Q[:, :, :nv].min(1), Q[:, :, :nv].max(1)], 1)
    raw = np.concatenate([S, agg], 1)
    raw = (raw - raw[tr].mean(0)) / (raw[tr].std(0) + 1e-6)
    Y = coh[OUTCOMES].apply(pd.to_numeric, errors="coerce").fillna(0).clip(0, 1).to_numpy(np.float32)
    log.info("stays %d | static %d cols | seq %s | outcome rates %s", len(coh), S.shape[1], Q.shape,
             dict(zip(OUTCOMES, Y.mean(0).round(3))))
    return Data(static=S, seq=Q, Y=Y, part=part, raw=raw.astype(np.float32),
                careunit=coh.first_careunit.fillna("unknown").to_numpy(), prep=prep)


def summary(d: Data) -> pd.DataFrame:
    rows = []
    for p in ("train", "val", "test"):
        m = d.part == p
        rows.append({"part": p, "stays": int(m.sum()), **{o: float(d.Y[m, j].mean()) for j, o in enumerate(OUTCOMES)}})
    return pd.DataFrame(rows)


In [ ]:
%%writefile /kaggle/working/repo/src/gcp.py
"""BigQuery helpers. Works with Kaggle's "Google Cloud Services" add-on
(linked account) and with normal ADC credentials elsewhere."""
from __future__ import annotations

import json
import logging
import os
from pathlib import Path

log = logging.getLogger(__name__)
SOURCE_PROJECT = "physionet-data"


def _kaggle_integrations() -> str:
    return os.environ.get("KAGGLE_KERNEL_INTEGRATIONS", "")


def credentials(target: str):
    """Credentials for 'bigquery' or 'gcs'.
    On Kaggle: the account attached in Add-ons -> Google Cloud Services (passed explicitly, so we never
    fall back to the GCE metadata server). Elsewhere: Application Default Credentials (None)."""
    if "KAGGLE_USER_SECRETS_TOKEN" not in os.environ:
        return None                                   # not on Kaggle -> ADC
    want = {"bigquery": "BIGQUERY", "gcs": "GCS"}[target]
    have = [x.upper() for x in _kaggle_integrations().split(":") if x]
    if want in have:
        from kaggle_secrets import GcpTarget
        try:
            from kaggle_gcp import KaggleKernelCredentials
        except ImportError:                           # minimal re-implementation
            from google.auth import credentials as gac
            from kaggle_secrets import UserSecretsClient

            class KaggleKernelCredentials(gac.Credentials):
                def __init__(self, target):
                    super().__init__()
                    self.target = target

                def refresh(self, request):
                    self.token, self.expiry = UserSecretsClient()._get_access_token(self.target)
        return KaggleKernelCredentials(target=GcpTarget[want])
    try:                                              # Add-ons -> "Google Cloud SDK" also works
        from google.oauth2.credentials import Credentials
        from kaggle_secrets import UserSecretsClient
        info = json.loads(UserSecretsClient().get_gcloud_credential())
        log.info("Using Google Cloud SDK add-on credential for %s", target)
        return Credentials.from_authorized_user_info(info)
    except Exception:
        pass
    raise RuntimeError(
        f"Kaggle has no {want} integration attached (KAGGLE_KERNEL_INTEGRATIONS='{_kaggle_integrations()}').\n"
        "Fix: Add-ons -> Google Cloud Services -> attach your Google account and tick BOTH 'BigQuery' and "
        "'Cloud Storage' -> Save -> then Run -> Restart & clear outputs -> Run All.")


def validate_project_id(project_id: str) -> str:
    pid = (project_id or "").strip()
    if not pid or "YOUR_" in pid.upper() or " " in pid:
        raise ValueError(
            f"cfg.project_id='{project_id}' is not a real GCP project ID.\n"
            "Find it: console.cloud.google.com -> project picker (top bar) -> copy the 'ID' column "
            "(e.g. 'mimic-research-123456', NOT the display name). Use the same Google account/project "
            "that is attached in Kaggle Add-ons and has billing/BigQuery enabled.")
    return pid


def check_access(project_id: str, probes=()):
    """Quick pre-flight check printed in the notebook."""
    try:
        validate_project_id(project_id)
        print("project_id:", project_id, "OK")
    except ValueError as e:
        print("project_id: FAILED ->", e)
        return
    print("Kaggle integrations:", _kaggle_integrations() or "(none)")
    for t in ("bigquery", "gcs"):
        try:
            c = credentials(t)
            if c is not None:
                from google.auth.transport.requests import Request
                c.refresh(Request())
            print(f"  {t:8s}: OK")
        except Exception as e:
            print(f"  {t:8s}: FAILED -> {e}")
            return
    try:
        client = bq_client(project_id)
        from google.cloud import bigquery
        client.query("SELECT 1", job_config=bigquery.QueryJobConfig(dry_run=True))
        print("  BigQuery project/billing: OK")
    except Exception as e:
        print("  BigQuery project/billing: FAILED ->", str(e)[:300])
        return
    for t in probes:
        ok = _dry_run_ok(client, f"SELECT 1 FROM `{SOURCE_PROJECT}.{t}` LIMIT 1")
        print(f"  PhysioNet {t}: {'OK' if ok else 'not readable (other version name is tried automatically)'}")


def bq_client(project_id: str):
    from google.cloud import bigquery
    project_id = validate_project_id(project_id)
    return bigquery.Client(project=project_id, credentials=credentials("bigquery"))


def _dry_run_ok(client, sql: str) -> bool:
    from google.api_core import exceptions as gexc
    from google.cloud import bigquery
    try:
        client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
        return True
    except (gexc.NotFound, gexc.Forbidden, gexc.BadRequest):
        return False            # dataset absent or no PhysioNet access -> try next candidate


def resolve_dataset(client, override: str, candidates: list[str], probe_table: str) -> str:
    """Return 'physionet-data.<dataset>' that the account can read (free dry-run probe)."""
    names = [override] if override else candidates
    for name in names:
        fq = name if "." in name else f"{SOURCE_PROJECT}.{name}"
        if _dry_run_ok(client, f"SELECT 1 FROM `{fq}.{probe_table}` LIMIT 1"):
            log.info("Using dataset %s", fq)
            return fq
    raise RuntimeError(
        f"None of {names} is readable (probe table '{probe_table}'). Open BigQuery console -> "
        f"'{SOURCE_PROJECT}' and set the dataset name in the config.")


def query_df(client, sql: str):
    job = client.query(sql)
    job.result()
    log.info("BigQuery processed %.2f GB", (job.total_bytes_processed or 0) / 1e9)
    try:
        return job.to_dataframe(create_bqstorage_client=True)
    except Exception as e:  # Storage Read API not permitted -> REST fallback
        log.warning("BQ Storage API unavailable (%s). Falling back to REST download.", type(e).__name__)
        return job.result().to_dataframe(create_bqstorage_client=False)


In [ ]:
%%writefile /kaggle/working/repo/src/metrics.py
"""Binary-classification metrics, bootstrap CIs, plots and small-cell suppression (PhysioNet rule of thumb:
never publish a count below 10)."""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.metrics import (average_precision_score, brier_score_loss, f1_score, precision_score, recall_score,
                             roc_auc_score, roc_curve)

MIN_CELL = 10


def ece(y, p, bins: int = 10) -> float:
    y, p = np.asarray(y, float), np.asarray(p, float)
    edges = np.quantile(p, np.linspace(0, 1, bins + 1))
    idx = np.clip(np.searchsorted(edges, p, side="right") - 1, 0, bins - 1)
    e = 0.0
    for b in range(bins):
        m = idx == b
        if m.any():
            e += m.mean() * abs(y[m].mean() - p[m].mean())
    return float(e)


def binary(y, p, thr: float | None = None) -> dict:
    y, p = np.asarray(y), np.asarray(p, float)
    out = dict(n=len(y), n_pos=int(y.sum()), prevalence=float(y.mean()) if len(y) else np.nan)
    if 0 < y.sum() < len(y):
        out.update(auroc=roc_auc_score(y, p), auprc=average_precision_score(y, p), brier=brier_score_loss(y, p),
                   ece=ece(y, p))
    if thr is not None:
        pr = p >= thr
        out.update(threshold=thr, f1=f1_score(y, pr, zero_division=0), precision=precision_score(y, pr, zero_division=0),
                   recall=recall_score(y, pr, zero_division=0),
                   specificity=float(((~pr) & (y == 0)).sum() / max((y == 0).sum(), 1)))
    return out


def bootstrap_ci(y, p, fn=roc_auc_score, n: int = 200, seed: int = 0, max_n: int = 200_000):
    """Percentile 95% CI. Large sets are subsampled to `max_n` per resample (CI is then slightly conservative)."""
    y, p = np.asarray(y), np.asarray(p)
    if n <= 0 or y.sum() == 0 or y.sum() == len(y):
        return (np.nan, np.nan)
    r = np.random.default_rng(seed)
    k = min(len(y), max_n)
    vals = []
    for _ in range(n):
        i = r.integers(0, len(y), k)
        if 0 < y[i].sum() < k:
            vals.append(fn(y[i], p[i]))
    return (float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))) if vals else (np.nan, np.nan)


def best_f1_threshold(y, p) -> float:
    ts = np.unique(np.quantile(p, np.linspace(0.01, 0.99, 199)))
    return float(ts[int(np.argmax([f1_score(y, p >= t, zero_division=0) for t in ts]))])


def suppress(df: pd.DataFrame, count_cols=("n", "n_pos"), min_n: int = MIN_CELL) -> pd.DataFrame:
    """Drop rows whose counts could identify few patients (n < 10). Zero positives is not a small cell."""
    df = df.copy()
    keep = np.ones(len(df), bool)
    for c in count_cols:
        if c in df:
            v = pd.to_numeric(df[c], errors="coerce").fillna(0).to_numpy()
            keep &= (v == 0) | (v >= min_n)
    return df[keep].reset_index(drop=True)


def plot_roc_pr(curves: dict, path, pos_label="positive"):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from sklearn.metrics import precision_recall_curve
    fig, ax = plt.subplots(1, 2, figsize=(10, 4.2))
    for name, (y, p) in curves.items():
        y, p = np.asarray(y), np.asarray(p)
        if not (0 < y.sum() < len(y)):
            continue
        fpr, tpr, _ = roc_curve(y, p)
        pr, rc, _ = precision_recall_curve(y, p)
        k = max(1, len(fpr) // 2000)
        ax[0].plot(fpr[::k], tpr[::k], label=f"{name} ({roc_auc_score(y, p):.3f})")
        k = max(1, len(rc) // 2000)
        ax[1].plot(rc[::k], pr[::k], label=f"{name} ({average_precision_score(y, p):.3f})")
    ax[0].plot([0, 1], [0, 1], "k--", lw=.7)
    ax[0].set(xlabel="False positive rate", ylabel="True positive rate", title="ROC (AUROC)")
    ax[1].set(xlabel=f"Recall ({pos_label})", ylabel=f"Precision ({pos_label})", title="Precision-recall (AUPRC)")
    for a in ax:
        a.legend(fontsize=7)
        a.grid(alpha=.3)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def plot_calibration(curves: dict, path, bins: int = 10):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(4.8, 4.4))
    for name, (y, p) in curves.items():
        y, p = np.asarray(y, float), np.asarray(p, float)
        q = np.quantile(p, np.linspace(0, 1, bins + 1))
        idx = np.clip(np.searchsorted(q, p, side="right") - 1, 0, bins - 1)
        xs = [p[idx == b].mean() for b in range(bins) if (idx == b).sum() >= MIN_CELL]
        ys = [y[idx == b].mean() for b in range(bins) if (idx == b).sum() >= MIN_CELL]
        ax.plot(xs, ys, "o-", ms=3, label=name)
    ax.plot([0, 1], [0, 1], "k--", lw=.7)
    ax.set(xlabel="Predicted probability", ylabel="Observed rate", title="Calibration (deciles)")
    ax.legend(fontsize=7)
    ax.grid(alpha=.3)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def md(df: pd.DataFrame, floatfmt="{:.3f}") -> str:
    if df is None or not len(df):
        return "–"

    def f(v):
        if isinstance(v, (float, np.floating)):
            return "–" if np.isnan(v) else floatfmt.format(v)
        if isinstance(v, (int, np.integer)):
            return f"{v:,}"
        return str(v)
    return "\n".join(["| " + " | ".join(map(str, df.columns)) + " |", "|" + "---|" * len(df.columns)] +
                     ["| " + " | ".join(f(v) for v in r) + " |" for r in df.itertuples(index=False)])


In [ ]:
%%writefile /kaggle/working/repo/src/model.py
"""Patient encoder: GRU over the first-24-h vitals + MLP over the static table -> 64-d embedding (L2-normalised for
search). Multi-task heads on the embedding force it to keep what matters for the outcomes. Resumable training."""
from __future__ import annotations

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score

from .trainer import Checkpointer, TensorBatches, fit


class Encoder(nn.Module):
    def __init__(self, d_seq, d_static, n_out, emb=64, gru=64, static_hidden=128, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(d_seq, gru, batch_first=True)
        self.static = nn.Sequential(nn.Linear(d_static, static_hidden), nn.ReLU(), nn.Dropout(dropout))
        self.proj = nn.Sequential(nn.Linear(gru + static_hidden, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, emb))
        self.heads = nn.Linear(emb, n_out)

    def embed(self, seq, static):
        return self.proj(torch.cat([self.gru(seq)[1][-1], self.static(static)], -1))

    def forward(self, seq, static):
        return self.heads(self.embed(seq, static))


@torch.no_grad()
def encode(model, d, idx, device, bs=8192):
    """-> (L2-normalised embeddings, head probabilities)"""
    model.eval()
    E, P = [], []
    for s in range(0, len(idx), bs):
        i = idx[s:s + bs]
        z = model.embed(torch.tensor(d.seq[i], device=device), torch.tensor(d.static[i], device=device))
        E.append(F.normalize(z, dim=-1).cpu().numpy())
        P.append(torch.sigmoid(model.heads(z)).cpu().numpy())
    return (np.concatenate(E).astype(np.float32), np.concatenate(P)) if E else (np.zeros((0, 1)), np.zeros((0, 1)))


def train(cfg, d, device, store, key):
    torch.manual_seed(cfg.seed)
    model = Encoder(d.seq.shape[2], d.static.shape[1], d.Y.shape[1], cfg.emb_dim, cfg.gru_hidden, cfg.static_hidden,
                    cfg.dropout).to(device)
    tr, va = d.idx("train"), d.idx("val")
    pos = d.Y[tr].mean(0).clip(1e-3, 1 - 1e-3)
    pw = torch.tensor(np.sqrt((1 - pos) / pos), device=device)          # mild re-weighting of rare outcomes
    batches = TensorBatches({"seq": d.seq[tr], "static": d.static[tr], "y": d.Y[tr]}, cfg.batch_size, cfg.seed, device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=1e-4)

    def loss_fn(m, b):
        return F.binary_cross_entropy_with_logits(m(b["seq"], b["static"]), b["y"], pos_weight=pw)

    def evaluate(m):
        _, p = encode(m, d, va, device)
        aucs = [roc_auc_score(d.Y[va, j], p[:, j]) for j in range(d.Y.shape[1]) if 0 < d.Y[va, j].sum() < len(va)]
        return {"val_mean_auroc": float(np.mean(aucs)) if aucs else float("nan")}
    return fit(model, opt, epochs=cfg.epochs, batches=batches, loss_fn=loss_fn, evaluate=evaluate,
               ckpt=Checkpointer(store, key, f"{cfg.work_dir}/ckpt/encoder"), metric="val_mean_auroc",
               patience=cfg.patience, ckpt_every_min=cfg.ckpt_every_min, log_every=0)


In [ ]:
%%writefile /kaggle/working/repo/src/pipeline.py
"""BigQuery (MIMIC-IV ICU stays, full) -> first-24-h inputs + later outcomes -> resumable encoder -> FAISS index of
past patients -> retrieval evaluation vs raw-feature / PCA search -> speed test -> demo query -> report + .pth (bucket only)."""
from __future__ import annotations

import hashlib
import json
import logging
import pickle
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from . import cohort, features, model as enc_model, search
from .bq import BigQuerySource, materialize, read
from .config import Config
from .metrics import md, suppress
from .trainer import history

log = logging.getLogger("similarity")

DEMO_PATIENT = {"age": 72, "female": 0, "sofa": 9, "cardiovascular": 3, "respiration": 2, "renal": 2,
                "creatinine_max": 2.4, "bun_max": 48, "wbc_max": 18, "platelets_min": 110, "gcs_min": 13,
                "hr": 112, "mbp": 62, "sbp": 92, "rr": 26, "spo2": 93, "temp": 38.4,
                "first_careunit": "Medical Intensive Care Unit (MICU)", "admission_type": "EW EMER."}


def run(cfg: Config, source=None) -> dict:
    t0 = time.time()
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True)
    for noisy in ("google", "urllib3", "matplotlib", "faiss"):
        logging.getLogger(noisy).setLevel(logging.WARNING)
    out = Path(cfg.work_dir)
    for sub in ("figures", "model", "report", "tables"):
        (out / sub).mkdir(parents=True, exist_ok=True)
    from .store import make_store
    store = make_store(cfg)
    src = source or BigQuerySource(cfg, store, cfg.datasets())
    device = torch.device(cfg.device or ("cuda" if torch.cuda.is_available() else "cpu"))
    log.info("device: %s | bucket: %s | FAISS: %s", device, store.uri(), search._faiss() is not None)

    # %% ICU stays from BigQuery | All adult MIMIC-IV ICU stays ≥ 24 h: first-24-h data (vitals hourly, first-day labs/SOFA/GCS/urine, Charlson) and outcomes after 24 h. Cached as Parquet in your bucket.
    # show: features.summary(d)
    coh = read(materialize(src, store, "sim_cohort", cohort.cohort_sql(cfg), cfg.cache_dir))
    hourly = read(materialize(src, store, "sim_vitals", cohort.hourly_sql(cfg), cfg.cache_dir))
    d = features.build(coh, hourly, cfg)
    fp = hashlib.sha1(json.dumps([len(d.Y), float(d.Y.sum()), list(d.static.shape), cfg.seed, cfg.emb_dim]).encode()).hexdigest()[:8]
    key = lambda name: f"ckpt/{cfg.run_name}-{fp}/{name}"

    # %% Train the patient encoder (resumable mid-epoch) | GRU (vitals) + MLP (static) → 64-d embedding with five outcome heads. Model, optimizer and the position inside the epoch are saved to the bucket every few minutes.
    # show: history(st, "val_mean_auroc")
    model, st = enc_model.train(cfg, d, device, store, key("encoder"))
    allidx = np.arange(len(d.Y))
    emb, head_p = enc_model.encode(model, d, allidx, device)

    # %% Build the FAISS index | Database = training patients (the "past patients"); queries = test patients, never seen in training.
    # show: pd.Series({"indexed patients": len(tr), "dimension": emb.shape[1], "backend": "FAISS" if index.faiss else "torch"})
    tr, te = d.idx("train"), d.idx("test")
    index = search.Index(emb[tr], "ip")

    # %% Retrieval evaluation | For each test patient, the outcome rate of its k nearest past patients is used as a risk estimate. Compared with search on raw features and on PCA; `lift_at_k` = how enriched the top neighbours of positive patients are.
    # show: res.pivot(index="outcome", columns="method", values="auroc")
    res = search.evaluate(d, emb, head_p, cfg)
    res.to_csv(out / "tables" / "retrieval.csv", index=False)
    search.plot_knn(res[~res.method.str.startswith("Encoder")], out / "figures" / "knn_auroc.png")
    search.plot_map(emb[te], d.Y[te, 0], out / "figures" / "embedding_map.png")

    # %% Speed: exact vs approximate index | FAISS IVF-PQ compresses each patient to a few bytes; recall@10 against exact search.
    # show: pd.Series(ann)
    ann = search.ann_benchmark(emb[tr], emb[te], cfg)

    # %% Demo query | A HYPOTHETICAL patient typed in by hand (not a real record). The answer only shows outcome rates among the most similar past patients.
    # show: demo_out
    demo_out = search.demo(DEMO_PATIENT, model, d.prep, index, d.Y[tr], cfg, device)

    # %% Export + report | Encoder `.pth` (with preprocessing) and the FAISS index go to /kaggle/working and your private bucket only: the index holds embeddings of real patients.
    # show: pd.Series(summary["models"])
    md_dir = out / "model"
    torch.save({"state_dict": model.state_dict(), "prep": pickle.dumps(d.prep), "d_seq": d.seq.shape[2],
                "d_static": d.static.shape[1], "outcomes": cohort.OUTCOMES, "emb_dim": cfg.emb_dim,
                "gru_hidden": cfg.gru_hidden, "static_hidden": cfg.static_hidden}, md_dir / "patient_encoder.pth")
    np.save(md_dir / "index_embeddings.npy", emb[tr])
    np.save(md_dir / "index_outcomes.npy", d.Y[tr])
    if index.faiss:
        import faiss
        faiss.write_index(index.ix, str(md_dir / "patients.faiss"))
    by_unit = []
    for u in np.unique(d.careunit[te]):
        m = d.careunit[te] == u
        if m.sum() >= 10:
            by_unit.append(dict(careunit=u, n=int(m.sum()), mortality=float(d.Y[te][m, 0].mean())))
    summary = dict(stays=len(d.Y), split=features.summary(d).to_dict("records"), encoder_best_epoch=st.best_epoch,
                   retrieval=res.to_dict("records"), ann=ann, demo=demo_out.to_dict(), demo_patient=DEMO_PATIENT,
                   test_units=suppress(pd.DataFrame(by_unit), ("n",)).to_dict("records"),
                   runtime_min=round((time.time() - t0) / 60, 1), config=cfg.to_dict(),
                   models={f.name: str(f) for f in sorted(md_dir.iterdir())})
    (out / "metrics.json").write_text(json.dumps(summary, indent=1, default=float))
    from .report import build as build_report
    build_report(out, res, summary)
    store.put_dir(out / "report", f"runs/{cfg.run_name}/report")
    store.put_dir(md_dir, f"runs/{cfg.run_name}/model")
    log.info("DONE in %.1f min", summary["runtime_min"])
    return summary


In [ ]:
%%writefile /kaggle/working/repo/src/report.py
from __future__ import annotations

import shutil
from pathlib import Path

import pandas as pd

from .metrics import md


def build(out: Path, res: pd.DataFrame, s: dict) -> str:
    rep = out / "report"
    (rep / "figures").mkdir(parents=True, exist_ok=True)
    for f in (out / "figures").glob("*.png"):
        shutil.copy(f, rep / "figures" / f.name)
    for f in (out / "tables").glob("*.csv"):
        shutil.copy(f, rep / f.name)
    c = s["config"]
    auroc = res.pivot(index="outcome", columns="method", values="auroc").reset_index()
    lift = res.dropna(subset=["lift_at_k"]).pivot(index="outcome", columns="method", values="lift_at_k").reset_index()
    demo_in = ", ".join(f"{k}={v}" for k, v in s["demo_patient"].items())
    demo = pd.DataFrame({"": list(s["demo"].keys()), "value": list(s["demo"].values())})
    text = f"""# Results — ICU patient similarity search (MIMIC-IV)

**Database:** {s['stays']:,} adult ICU stays ≥ {c['seq_hours']} h (full MIMIC-IV), split by patient
{'/'.join(str(int(x * 100)) for x in c['split'])}. Each stay becomes a {c['emb_dim']}-d embedding of its first
{c['seq_hours']} h (hourly vitals + first-day labs, SOFA, GCS, urine, weight, Charlson). The training split is the
searchable "past patients" index (FAISS, cosine); test patients are the queries.
**Outcomes (after the first 24 h):** in-hospital death, ICU stay > 7 days, invasive ventilation, vasopressor,
AKI stage ≥ 2 (days 2–7).

## Data
{md(pd.DataFrame(s['split']))}

## Do similar past patients predict what happens next?
Outcome rate among the k = {c['k']} nearest past patients, used as a risk score → AUROC on test queries.
"Encoder heads" = the network's own direct prediction (upper reference).

{md(auroc)}

![kNN AUROC](figures/knn_auroc.png)

**Lift@{c['k_lift']}:** for test patients who had the outcome, how many times more often their {c['k_lift']} nearest
neighbours had it than a random past patient.

{md(lift)}

## Embedding space
![map](figures/embedding_map.png)

## Speed (FAISS)
{md(pd.DataFrame([s['ann']]))}

## Demo query (hypothetical patient, not a real record)
Input: {demo_in}

{md(demo)}

---
Encoder best epoch {s['encoder_best_epoch']} · runtime {s['runtime_min']} min. Aggregate results only; the index and the
encoder stay private (the index holds embeddings of real patients). This is a decision-support prototype, not a
clinical tool.
"""
    (rep / "REPORT.md").write_text(text, encoding="utf-8")
    return text


In [ ]:
%%writefile /kaggle/working/repo/src/search.py
"""Similarity search with FAISS (exact inner product on L2-normalised embeddings = cosine), a torch fallback when FAISS
is missing, retrieval evaluation, an approximate-index speed test and a privacy-safe demo query."""
from __future__ import annotations

import time

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score

from .cohort import OUTCOMES, VITALS


def _faiss():
    try:
        import faiss
        return faiss
    except ImportError:
        return None


class Index:
    def __init__(self, base: np.ndarray, metric: str = "ip"):
        self.metric, self.base = metric, np.ascontiguousarray(base, dtype=np.float32)
        f = _faiss()
        self.faiss = f is not None
        if self.faiss:
            self.ix = (f.IndexFlatIP if metric == "ip" else f.IndexFlatL2)(self.base.shape[1])
            self.ix.add(self.base)

    def search(self, Q: np.ndarray, k: int):
        Q = np.ascontiguousarray(Q, dtype=np.float32)
        if self.faiss:
            return self.ix.search(Q, k)
        b = torch.from_numpy(self.base)
        D, I = [], []
        for s in range(0, len(Q), 4096):
            q = torch.from_numpy(Q[s:s + 4096])
            sim = q @ b.T if self.metric == "ip" else -torch.cdist(q, b)
            v, i = sim.topk(k, dim=1)
            D.append((v if self.metric == "ip" else -v).numpy())
            I.append(i.numpy())
        return np.concatenate(D), np.concatenate(I)


def knn_eval(base, Yb, query, Yq, k, k_lift, metric) -> list[dict]:
    ix = Index(base, metric)
    _, I = ix.search(query, k)
    risk = Yb[I].mean(1)                                       # neighbour outcome rate = predicted risk
    rows = []
    for j, o in enumerate(OUTCOMES):
        y = Yq[:, j]
        if 0 < y.sum() < len(y):
            base_rate = Yb[:, j].mean()
            lift = Yb[I[y == 1, :k_lift], j].mean() / max(base_rate, 1e-9)
            rows.append(dict(outcome=o, auroc=roc_auc_score(y, risk[:, j]), lift_at_k=lift))
    return rows


def evaluate(d, emb, head_p, cfg) -> pd.DataFrame:
    from sklearn.decomposition import PCA
    tr, te = d.idx("train"), d.idx("test")
    pca = PCA(n_components=min(cfg.emb_dim, d.raw.shape[1]), random_state=cfg.seed).fit(d.raw[tr])
    spaces = {"Learned embedding (cosine)": (emb[tr], emb[te], "ip"),
              "Raw features (Euclidean)": (d.raw[tr], d.raw[te], "l2"),
              f"PCA-{pca.n_components_} of raw features": (pca.transform(d.raw[tr]), pca.transform(d.raw[te]), "l2")}
    rows = []
    for name, (b, q, metric) in spaces.items():
        for r in knn_eval(b, d.Y[tr], q, d.Y[te], cfg.k, cfg.k_lift, metric):
            rows.append(dict(method=name, **r))
    for j, o in enumerate(OUTCOMES):                         # reference: the encoder's own prediction heads
        y = d.Y[te, j]
        if 0 < y.sum() < len(y):
            rows.append(dict(method="Encoder heads (direct prediction)", outcome=o, auroc=roc_auc_score(y, head_p[te, j]),
                             lift_at_k=np.nan))
    return pd.DataFrame(rows)


def ann_benchmark(base, query, cfg) -> dict:
    """Exact vs approximate (IVF-PQ) FAISS: recall@10 and milliseconds per query."""
    f = _faiss()
    if f is None or len(base) < cfg.ivf_nlist * 40:
        return {"note": "FAISS not installed or database too small for IVF-PQ; exact search used"}
    base, query = np.ascontiguousarray(base, np.float32), np.ascontiguousarray(query[:5000], np.float32)
    flat = f.IndexFlatIP(base.shape[1])
    flat.add(base)
    t = time.time()
    _, I0 = flat.search(query, 10)
    t_flat = (time.time() - t) / len(query) * 1000
    q = f.IndexFlatIP(base.shape[1])
    ivf = f.IndexIVFPQ(q, base.shape[1], cfg.ivf_nlist, cfg.ivf_pq_m, 8, f.METRIC_INNER_PRODUCT)
    ivf.train(base)
    ivf.add(base)
    ivf.nprobe = 16
    t = time.time()
    _, I1 = ivf.search(query, 10)
    t_ivf = (time.time() - t) / len(query) * 1000
    rec = np.mean([len(set(a) & set(b)) / 10 for a, b in zip(I0, I1)])
    return {"database_size": len(base), "exact_ms_per_query": round(t_flat, 4), "ivfpq_ms_per_query": round(t_ivf, 4),
            "ivfpq_recall_at_10": round(float(rec), 4), "ivfpq_bytes_per_patient": cfg.ivf_pq_m}


@torch.no_grad()
def demo(profile: dict, model, prep, index: Index, Yb: np.ndarray, cfg, device, k=None) -> pd.Series:
    """Encode a HYPOTHETICAL patient (typed-in values; vitals held constant for 24 h) and summarise the outcomes of
    the k most similar past patients. Only aggregates over >= 10 patients are shown — no individual records."""
    import torch.nn.functional as F
    k = max(10, k or cfg.k)
    row = pd.DataFrame([{c: profile.get(c, np.nan) for c in profile}])
    for c in ("first_careunit", "admission_type"):
        row[c] = profile.get(c, "unknown")
    cube = {v: np.full((1, cfg.seq_hours), profile.get(v, np.nan), float) for v in VITALS}
    S, Q = prep.static(row), prep.sequence(cube)
    model.eval()
    z = F.normalize(model.embed(torch.tensor(Q, device=device), torch.tensor(S, device=device)), dim=-1).cpu().numpy()
    sim, I = index.search(z, k)
    out = {f"{o} among {k} most similar": float(Yb[I[0], j].mean()) for j, o in enumerate(OUTCOMES)}
    out["mean cosine similarity"] = float(sim[0].mean())
    return pd.Series(out)


def plot_knn(res: pd.DataFrame, path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    w = res.pivot(index="outcome", columns="method", values="auroc").reindex(OUTCOMES)
    ax = w.plot.bar(figsize=(9, 4.2), width=.8)
    ax.set(ylabel="AUROC (test queries)", xlabel="", title=f"Outcome of the k nearest past patients as a prediction",
           ylim=(0.5, 1.0))
    ax.legend(fontsize=7)
    ax.tick_params(axis="x", rotation=15)
    ax.grid(alpha=.3, axis="y")
    ax.figure.tight_layout()
    ax.figure.savefig(path, dpi=150)
    plt.close(ax.figure)


def plot_map(emb_test, y_death, path):
    """2-D PCA of the test embeddings; each hexagon = >= 10 patients, colour = in-hospital mortality."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from sklearn.decomposition import PCA
    xy = PCA(2, random_state=0).fit_transform(emb_test)
    fig, ax = plt.subplots(figsize=(5.6, 4.8))
    hb = ax.hexbin(xy[:, 0], xy[:, 1], C=y_death, reduce_C_function=np.mean, gridsize=30, mincnt=10, cmap="viridis")
    fig.colorbar(hb, label="in-hospital mortality")
    ax.set(xlabel="PC 1", ylabel="PC 2", title="Embedding space (test patients, hexagons ≥ 10)")
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


In [ ]:
%%writefile /kaggle/working/repo/src/store.py
"""Checkpoint store: a private GCS bucket (auto-created) or a local folder (tests / no GCP).
Everything needed to resume after a Kaggle session stops lives here."""
from __future__ import annotations

import logging
import re
import shutil
from pathlib import Path

log = logging.getLogger(__name__)


def default_bucket_name(project_id: str) -> str:
    name = re.sub(r"[^a-z0-9-]", "-", f"{project_id}-mimic-ckpt".lower()).strip("-")
    return name[:63]


class GCSStore:
    def __init__(self, project_id: str, bucket: str, prefix: str, location: str = "US"):
        from google.cloud import storage

        from .gcp import credentials
        self.client = storage.Client(project=project_id, credentials=credentials("gcs"))
        self.name = (bucket or default_bucket_name(project_id)).replace("gs://", "").strip("/")
        self.prefix = prefix.strip("/")
        self.bucket = self._ensure(project_id, location)

    def _ensure(self, project_id, location):
        from google.api_core import exceptions as gexc
        b = self.client.bucket(self.name)
        try:
            if b.exists():
                return b
        except gexc.Forbidden:
            return b           # exists but owned elsewhere / no get permission; writes will tell
        try:
            b.iam_configuration.uniform_bucket_level_access_enabled = True
            b.iam_configuration.public_access_prevention = "enforced"   # never public
            b = self.client.create_bucket(b, project=project_id, location=location)
            log.info("Created private bucket gs://%s (%s)", self.name, location)
        except gexc.Conflict:
            raise RuntimeError(f"Bucket name '{self.name}' is taken globally. Set cfg.bucket to another name.")
        except Exception as e:
            raise RuntimeError(
                f"Could not create bucket gs://{self.name} ({e}). Create it once in Cloud Shell:\n"
                f"  gcloud storage buckets create gs://{self.name} --project={project_id} --location={location} "
                f"--uniform-bucket-level-access --public-access-prevention\nthen set cfg.bucket='{self.name}'.")
        return b

    def uri(self, key=""):
        return f"gs://{self.name}/{self.prefix}/{key}".rstrip("/")

    def _k(self, key):
        return f"{self.prefix}/{key}"

    def put(self, local, key):
        self.bucket.blob(self._k(key)).upload_from_filename(str(local), timeout=600)

    def get(self, key, local) -> bool:
        bl = self.bucket.blob(self._k(key))
        if not bl.exists():
            return False
        Path(local).parent.mkdir(parents=True, exist_ok=True)
        tmp = Path(str(local) + ".part")
        bl.download_to_filename(str(tmp), timeout=600)
        tmp.replace(local)                        # never leave a half-downloaded file behind
        return True

    def exists(self, key) -> bool:
        return self.bucket.blob(self._k(key)).exists()

    def list(self, sub) -> list[str]:
        p = self._k(sub).rstrip("/") + "/"
        return [b.name[len(self.prefix) + 1:] for b in self.client.list_blobs(self.bucket, prefix=p)]

    def put_text(self, text, key):
        self.bucket.blob(self._k(key)).upload_from_string(text)

    def get_text(self, key):
        bl = self.bucket.blob(self._k(key))
        return bl.download_as_text() if bl.exists() else None

    def put_dir(self, local_dir, sub):
        n = 0
        for p in sorted(Path(local_dir).rglob("*")):
            if p.is_file():
                self.put(p, f"{sub.strip('/')}/{p.relative_to(local_dir).as_posix()}")
                n += 1
        log.info("Uploaded %d files -> %s", n, self.uri(sub))


class LocalStore:
    """Same interface backed by a folder (unit tests, or running without GCP)."""

    def __init__(self, root, prefix=""):
        self.root = Path(root) / prefix
        self.root.mkdir(parents=True, exist_ok=True)
        self.name = str(self.root)

    def uri(self, key=""):
        return str(self.root / key)

    def put(self, local, key):
        dst = self.root / key
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(local, dst)

    def get(self, key, local) -> bool:
        src = self.root / key
        if not src.exists():
            return False
        Path(local).parent.mkdir(parents=True, exist_ok=True)
        if Path(local).resolve() != src.resolve():
            shutil.copyfile(src, local)
        return True

    def exists(self, key) -> bool:
        return (self.root / key).exists()

    def list(self, sub) -> list[str]:
        d = self.root / sub
        return [p.relative_to(self.root).as_posix() for p in d.rglob("*") if p.is_file()] if d.exists() else []

    def put_text(self, text, key):
        dst = self.root / key
        dst.parent.mkdir(parents=True, exist_ok=True)
        dst.write_text(text)

    def get_text(self, key):
        f = self.root / key
        return f.read_text() if f.exists() else None

    def put_dir(self, local_dir, sub):
        for p in Path(local_dir).rglob("*"):
            if p.is_file():
                self.put(p, f"{sub.strip('/')}/{p.relative_to(local_dir).as_posix()}")


def make_store(cfg):
    if getattr(cfg, "local_store", ""):
        return LocalStore(cfg.local_store, cfg.gcs_prefix)
    from .gcp import validate_project_id
    cfg.project_id = validate_project_id(cfg.project_id)
    s = GCSStore(cfg.project_id, cfg.bucket, cfg.gcs_prefix, cfg.bucket_location)
    cfg.bucket = s.name
    log.info("Checkpoints -> %s", s.uri())
    return s


In [ ]:
%%writefile /kaggle/working/repo/src/trainer.py
"""Resumable PyTorch training.

Everything needed to continue is saved as `last.pt` (model, optimizer, scheduler, RNG states, epoch AND the step
inside the epoch) every `ckpt_every_min` minutes and at the end of every epoch, locally and in the bucket.
The batch order of an epoch depends only on (seed, epoch), so a new Kaggle session skips exactly the batches that
were already done and continues in the middle of the epoch. `best.pt` keeps the best validation weights.
"""
from __future__ import annotations

import logging
import math
import time
import uuid
from dataclasses import asdict, dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd
import torch

log = logging.getLogger(__name__)


class Checkpointer:
    def __init__(self, store, key: str, local_dir):
        self.store, self.key, self.dir = store, key.strip("/"), Path(local_dir)
        self.dir.mkdir(parents=True, exist_ok=True)

    def save(self, obj, name: str):
        f = self.dir / name
        tmp = f.with_suffix(".tmp")
        torch.save(obj, tmp)
        tmp.replace(f)
        if self.store is not None:
            self.store.put(f, f"{self.key}/{name}")

    def load(self, name: str):
        f = self.dir / name
        if not f.exists() and self.store is not None:
            self.store.get(f"{self.key}/{name}", f)
        return torch.load(f, map_location="cpu", weights_only=False) if f.exists() else None


@dataclass
class State:
    epoch: int = 0                 # finished epochs
    step: int = 0                  # finished steps inside the current epoch
    loss_sum: float = 0.0
    loss_n: int = 0
    best: float = -math.inf
    best_epoch: int = 0
    bad_epochs: int = 0
    done: bool = False
    history: list = field(default_factory=list)


def _rng_state():
    s = {"torch": torch.get_rng_state(), "numpy": np.random.get_state()}
    if torch.cuda.is_available():
        s["cuda"] = torch.cuda.get_rng_state_all()
    return s


def _set_rng(s):
    torch.set_rng_state(s["torch"])
    np.random.set_state(s["numpy"])
    if "cuda" in s and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(s["cuda"])


class TensorBatches:
    """Mini-batches from in-memory arrays. Order = f(seed, epoch) -> exact mid-epoch resume.
    With `on_device=True` the arrays are copied to the GPU once (fast for tabular data)."""

    def __init__(self, arrays: dict, batch_size: int, seed: int, device, on_device: bool = True, index=None,
                 tensors: dict | None = None):
        """`index`: use only these rows (no copy of the big arrays). `tensors`: already-built torch tensors."""
        self.bs, self.seed, self.device = batch_size, seed, torch.device(device)
        tgt = self.device if on_device else torch.device("cpu")
        def to_t(v):                                   # numpy array OR torch tensor (CPU or GPU)
            if isinstance(v, torch.Tensor):
                return v.to(tgt)
            return torch.as_tensor(np.ascontiguousarray(v)).to(tgt)
        self.t = tensors if tensors is not None else {k: to_t(v) for k, v in arrays.items()}
        some = next(iter(self.t.values()))
        self.index = None if index is None else torch.as_tensor(np.asarray(index, dtype=np.int64)).to(some.device)
        self.n = len(some) if index is None else len(self.index)
        self._perm = (None, None)

    def __len__(self):
        return max(1, -(-self.n // self.bs))

    def __call__(self, epoch: int, step: int) -> dict:
        if self._perm[0] != epoch:
            self._perm = (epoch, torch.from_numpy(np.random.default_rng(self.seed + epoch).permutation(self.n)))
        idx = self._perm[1][step * self.bs:(step + 1) * self.bs]
        some = next(iter(self.t.values()))
        idx = idx.to(some.device)
        if self.index is not None:
            idx = self.index[idx]
        return {k: v[idx].to(self.device, non_blocking=True) for k, v in self.t.items()}


def fit(model, opt, *, epochs: int, batches, loss_fn, evaluate, ckpt: Checkpointer, metric: str,
        sched=None, patience: int = 0, ckpt_every_min: float = 10.0, grad_clip: float = 1.0, log_every: int = 500):
    """Train with resume. `evaluate(model) -> dict` must contain `metric` (higher is better).
    Returns (model with best weights, State)."""
    session = uuid.uuid4().hex[:6]
    st = State()
    last = ckpt.load("last.pt")
    if last is not None:
        model.load_state_dict(last["model"])
        opt.load_state_dict(last["opt"])
        if sched is not None and last.get("sched") is not None:
            sched.load_state_dict(last["sched"])
        st = State(**last["state"])
        _set_rng(last["rng"])
        log.info("RESUME from bucket checkpoint: %d epoch(s) done, %d/%d steps of epoch %d already done",
                 st.epoch, st.step, len(batches), st.epoch + 1)

    def save_last():
        ckpt.save({"model": model.state_dict(), "opt": opt.state_dict(),
                   "sched": sched.state_dict() if sched is not None else None,
                   "state": asdict(st), "rng": _rng_state()}, "last.pt")

    t_save = time.time()
    while not st.done and st.epoch < epochs:
        model.train()
        t_ep = time.time()
        for step in range(st.step, len(batches)):
            b = batches(st.epoch, step)
            loss = loss_fn(model, b)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            if grad_clip:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step()
            if sched is not None:
                sched.step()
            st.step, st.loss_sum, st.loss_n = step + 1, st.loss_sum + float(loss.detach()), st.loss_n + 1
            if log_every and (step + 1) % log_every == 0:
                log.info("epoch %d step %d/%d loss %.4f", st.epoch + 1, step + 1, len(batches), st.loss_sum / st.loss_n)
            if time.time() - t_save > ckpt_every_min * 60:
                save_last()
                t_save = time.time()
                log.info("mid-epoch checkpoint saved (epoch %d, step %d)", st.epoch + 1, st.step)
        model.eval()
        with torch.no_grad():
            val = {k: float(v) for k, v in evaluate(model).items()}
        row = {"epoch": st.epoch + 1, "train_loss": st.loss_sum / max(st.loss_n, 1), **val,
               "minutes": round((time.time() - t_ep) / 60, 2), "session": session}
        st.history.append(row)
        if val[metric] > st.best:
            st.best, st.best_epoch, st.bad_epochs = val[metric], st.epoch + 1, 0
            ckpt.save({"model": model.state_dict(), "epoch": st.epoch + 1, metric: val[metric]}, "best.pt")
        else:
            st.bad_epochs += 1
        log.info("epoch %d: %s", st.epoch + 1, ", ".join(f"{k}={v:.4f}" for k, v in row.items()
                                                       if isinstance(v, float)))
        st.epoch, st.step, st.loss_sum, st.loss_n = st.epoch + 1, 0, 0.0, 0
        if patience and st.bad_epochs >= patience:
            log.info("early stop: no %s improvement for %d epochs", metric, patience)
            st.done = True
        save_last()
        t_save = time.time()
    st.done = True
    save_last()
    best = ckpt.load("best.pt")
    if best is not None:
        model.load_state_dict(best["model"])
    model.eval()
    st.session = session
    return model, st


def history(st: State, metric: str) -> pd.DataFrame:
    """One row per epoch; `source` tells whether the epoch was trained now or restored from the bucket."""
    df = pd.DataFrame(st.history)
    if not len(df):
        return pd.DataFrame({"info": ["no epochs"]})
    cur = getattr(st, "session", None)
    df["source"] = np.where(df.session == cur, "this session", "checkpoint")
    df["best"] = np.where(df.epoch == st.best_epoch, "★", "")
    return df.drop(columns="session")


def cosine_warmup(opt, total_steps: int, warmup: float = 0.03):
    w = max(1, int(total_steps * warmup))

    def f(s):
        if s < w:
            return (s + 1) / w
        return 0.5 * (1 + math.cos(math.pi * min(1.0, (s - w) / max(1, total_steps - w))))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)


## 3 · Config
Change only this cell.

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/repo")
from src.config import Config

cfg = Config(
    project_id="mimic-cxr-training",   # <-- your GCP project ID (billing + BigQuery); bucket is created automatically
    work_dir="/kaggle/working/similarity",
    emb_dim=64, epochs=40,
    k=50,                              # neighbours used for the outcome summary
    run_name="main",                   # change to start a fresh set of checkpoints
)
cfg

## 4 · Pre-flight check
Project ID, Kaggle → Google Cloud add-on (BigQuery + Cloud Storage), billing and PhysioNet access. Costs nothing (dry runs).

In [ ]:
from src.gcp import check_access
check_access(cfg.project_id, probes=['mimiciv_3_1_derived.icustay_detail', 'mimiciv_derived.icustay_detail', 'mimiciv_3_1_icu.icustays', 'mimiciv_icu.icustays'])

## 5 · Load the pipeline
Imports the project code written above. The cells below are the steps of `src/pipeline.py::run()`, one per cell.

In [ ]:
import sys, time
from IPython.display import display
for _m in [m for m in list(sys.modules) if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]                       # always use the freshly written code (no stale imports)
import src.pipeline as _pipeline
globals().update({k: v for k, v in vars(_pipeline).items() if not k.startswith("__")})   # helpers + imports


class StopRun(Exception):
    """Raised on purpose (e.g. no GPU yet / paused): Run All again later to continue."""


def _stop(result):
    print(result)
    raise StopRun(result.get("status", "stopped") if isinstance(result, dict) else str(result))


source = None


## 6 · Start
Logging, folders and the checkpoint store.

In [ ]:
t0 = time.time()
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True)
for noisy in ("google", "urllib3", "matplotlib", "faiss"):
    logging.getLogger(noisy).setLevel(logging.WARNING)
out = Path(cfg.work_dir)
for sub in ("figures", "model", "report", "tables"):
    (out / sub).mkdir(parents=True, exist_ok=True)
from src.store import make_store
store = make_store(cfg)
src = source or BigQuerySource(cfg, store, cfg.datasets())
device = torch.device(cfg.device or ("cuda" if torch.cuda.is_available() else "cpu"))
log.info("device: %s | bucket: %s | FAISS: %s", device, store.uri(), search._faiss() is not None)

## 7 · ICU stays from BigQuery
All adult MIMIC-IV ICU stays ≥ 24 h: first-24-h data (vitals hourly, first-day labs/SOFA/GCS/urine, Charlson) and outcomes after 24 h. Cached as Parquet in your bucket.

In [ ]:
coh = read(materialize(src, store, "sim_cohort", cohort.cohort_sql(cfg), cfg.cache_dir))
hourly = read(materialize(src, store, "sim_vitals", cohort.hourly_sql(cfg), cfg.cache_dir))
d = features.build(coh, hourly, cfg)
fp = hashlib.sha1(json.dumps([len(d.Y), float(d.Y.sum()), list(d.static.shape), cfg.seed, cfg.emb_dim]).encode()).hexdigest()[:8]
key = lambda name: f"ckpt/{cfg.run_name}-{fp}/{name}"
display(features.summary(d))

## 8 · Train the patient encoder (resumable mid-epoch)
GRU (vitals) + MLP (static) → 64-d embedding with five outcome heads. Model, optimizer and the position inside the epoch are saved to the bucket every few minutes.

In [ ]:
model, st = enc_model.train(cfg, d, device, store, key("encoder"))
allidx = np.arange(len(d.Y))
emb, head_p = enc_model.encode(model, d, allidx, device)
display(history(st, "val_mean_auroc"))

## 9 · Build the FAISS index
Database = training patients (the "past patients"); queries = test patients, never seen in training.

In [ ]:
tr, te = d.idx("train"), d.idx("test")
index = search.Index(emb[tr], "ip")
display(pd.Series({"indexed patients": len(tr), "dimension": emb.shape[1], "backend": "FAISS" if index.faiss else "torch"}))

## 10 · Retrieval evaluation
For each test patient, the outcome rate of its k nearest past patients is used as a risk estimate. Compared with search on raw features and on PCA; `lift_at_k` = how enriched the top neighbours of positive patients are.

In [ ]:
res = search.evaluate(d, emb, head_p, cfg)
res.to_csv(out / "tables" / "retrieval.csv", index=False)
search.plot_knn(res[~res.method.str.startswith("Encoder")], out / "figures" / "knn_auroc.png")
search.plot_map(emb[te], d.Y[te, 0], out / "figures" / "embedding_map.png")
display(res.pivot(index="outcome", columns="method", values="auroc"))

## 11 · Speed: exact vs approximate index
FAISS IVF-PQ compresses each patient to a few bytes; recall@10 against exact search.

In [ ]:
ann = search.ann_benchmark(emb[tr], emb[te], cfg)
display(pd.Series(ann))

## 12 · Demo query
A HYPOTHETICAL patient typed in by hand (not a real record). The answer only shows outcome rates among the most similar past patients.

In [ ]:
demo_out = search.demo(DEMO_PATIENT, model, d.prep, index, d.Y[tr], cfg, device)
display(demo_out)

## 13 · Export + report
Encoder `.pth` (with preprocessing) and the FAISS index go to /kaggle/working and your private bucket only: the index holds embeddings of real patients.

In [ ]:
md_dir = out / "model"
torch.save({"state_dict": model.state_dict(), "prep": pickle.dumps(d.prep), "d_seq": d.seq.shape[2],
            "d_static": d.static.shape[1], "outcomes": cohort.OUTCOMES, "emb_dim": cfg.emb_dim,
            "gru_hidden": cfg.gru_hidden, "static_hidden": cfg.static_hidden}, md_dir / "patient_encoder.pth")
np.save(md_dir / "index_embeddings.npy", emb[tr])
np.save(md_dir / "index_outcomes.npy", d.Y[tr])
if index.faiss:
    import faiss
    faiss.write_index(index.ix, str(md_dir / "patients.faiss"))
by_unit = []
for u in np.unique(d.careunit[te]):
    m = d.careunit[te] == u
    if m.sum() >= 10:
        by_unit.append(dict(careunit=u, n=int(m.sum()), mortality=float(d.Y[te][m, 0].mean())))
summary = dict(stays=len(d.Y), split=features.summary(d).to_dict("records"), encoder_best_epoch=st.best_epoch,
               retrieval=res.to_dict("records"), ann=ann, demo=demo_out.to_dict(), demo_patient=DEMO_PATIENT,
               test_units=suppress(pd.DataFrame(by_unit), ("n",)).to_dict("records"),
               runtime_min=round((time.time() - t0) / 60, 1), config=cfg.to_dict(),
               models={f.name: str(f) for f in sorted(md_dir.iterdir())})
(out / "metrics.json").write_text(json.dumps(summary, indent=1, default=float))
from src.report import build as build_report
build_report(out, res, summary)
store.put_dir(out / "report", f"runs/{cfg.run_name}/report")
store.put_dir(md_dir, f"runs/{cfg.run_name}/model")
log.info("DONE in %.1f min", summary["runtime_min"])
result = summary
result
display(pd.Series(summary["models"]))

## 14 · Results report
Aggregate results only (safe for GitHub). Model weights stay in `/kaggle/working` and your private bucket.

In [ ]:
import re, shutil, glob
from IPython.display import Markdown, Image, display
rep = f"{cfg.work_dir}/report"
text = open(f"{rep}/REPORT.md", encoding="utf-8").read()
display(Markdown(re.sub(r"!\[.*?\]\(figures/.*?\)", "", text)))
for f in sorted(glob.glob(f"{rep}/figures/*.png")):
    display(Image(f))
shutil.make_archive("/kaggle/working/results_for_github", "zip", rep)
print("Results zip : /kaggle/working/results_for_github.zip  -> unzip into the repo as results/")
print("Models (.pth, private — keep out of GitHub):")
for f in sorted(glob.glob(f"{cfg.work_dir}/model/*.pth")):
    print("  ", f)
print("Bucket copy :", "gs://" + cfg.bucket + "/" + cfg.gcs_prefix + "/runs/" + cfg.run_name + "/model/")